## YOLOv8 Training for Coin Detection
This notebook demonstrates the process of training a YOLOv8 model for coin detection. It includes dataset preparation, hyperparameter tuning, model training, and inference on test images. The notebook is structured to provide a clear workflow for training a custom object detection model using the Ultralytics YOLO library

In [2]:
# ========== Import All Required Libraries ==========
from pathlib import Path
from ultralytics import YOLO
import os
import splitfolders
import yaml

## Dataset Preparation
Split the dataset into training, validation, and test sets using a 70-15-15 ratio. The dataset is assumed to be organized in a folder named "datasets".

In [ ]:
splitfolders.ratio(input="datasets",
                   output="data",
                   ratio=(0.7, 0.15, 0.15),
                   seed=42)


## Configuration Setup
Define all configuration variables including model paths, dataset locations, and hyperparameter files for the entire notebook.


In [ ]:
# ========== Configuration Variables ==========
# Model and paths configuration
MODEL_VERSION = "yolo26s.pt"
YAML_PATH = "../datasets/euro_coins/data.yaml"
BASE_RUNS_DIR = Path("runs") / "detect"
HYPERPARAMS_YAML = BASE_RUNS_DIR / "coin_detector" / "yolo26n_coin_tune" / "best_hyperparameters.yaml"
BEST_MODEL_PATH = BASE_RUNS_DIR / "train" / "weights" / "yolo26s_pre_trained.pt"
TEST_FOLDER = "test_img"

# Initialize model
model = YOLO(MODEL_VERSION)


## Hyperparameter Search Space
Define the search space for hyperparameter tuning. This includes augmentation parameters, learning rates, and regularization settings.


In [ ]:
search_space = {
    "lr0": (1e-4, 5e-3),
    "lrf": (0.05, 0.5),
    "weight_decay": (1e-5, 1e-3),

    "mosaic": (0.4, 0.9),
    "mixup": (0.0, 0.10),
    "close_mosaic": (10.0, 20.0),

    "degrees": (5.0, 20.0),
    "scale": (0.3, 0.7),
    "translate": (0.03, 0.15),
    "shear": (0.0, 4.0),
    "perspective": (0.0, 0.001),

    "hsv_h": (0.005, 0.03),
    "hsv_s": (0.3, 0.8),
    "hsv_v": (0.2, 0.5),
    "fliplr": (0.2, 0.6),
    "flipud": (0.0, 0.15),
}


## Hyperparameter Tuning
Run the YOLO hyperparameter tuning process to find optimal parameters for the coin detector model.


In [ ]:
result = model.tune(
    data=YAML_PATH,
    epochs=10, # Epochs Per Iterations
    iterations=5, # Total Number of Iterations

    imgsz=640,
    batch=32,
    device=0,
    cache="ram",
    workers=4,
    amp=True,
    optimizer="AdamW",

    project="coin_detector",
    name="yolo26n_coin_tune",

    space=search_space, # Param Grid
    plots=False,
    save=False,
    val=True,
)


## Train with Best Hyperparameters
Load the best hyperparameters from tuning results and train the model for full epochs with optimized settings.


In [ ]:
# Load best hyperparameters from tuning results
model = YOLO(MODEL_VERSION)  # Reload with fresh model

with open(HYPERPARAMS_YAML) as f:
    best_hyp = yaml.safe_load(f)

print(f"Training model with best hyperparameters from: {HYPERPARAMS_YAML}")
model.train(
    data=YAML_PATH,
    epochs=150,
    **best_hyp,
    verbose=True
)


## Load Best Trained Model
Load the best model weights that were saved during training for inference and evaluation.


In [ ]:
# Load best trained model
best_model = YOLO(str(BEST_MODEL_PATH))
print(f"Loaded best model from: {BEST_MODEL_PATH}")


## Run Inference on Test Images
Execute model predictions on test images stored in the test folder. Results are saved for evaluation.


In [ ]:
# Run inference on test images
print(f"Running inference on images in: {TEST_FOLDER}")
test_folder_path = Path(TEST_FOLDER)

if not test_folder_path.exists():
    print(f"Error: Test folder not found at {test_folder_path}")
else:
    image_count = 0
    for file in os.listdir(test_folder_path):
        if file.lower().endswith((".jpg", ".png", ".jpeg")):
            img_path = test_folder_path / file
            print(f"Processing: {img_path}")
            results = best_model.predict(str(img_path), save=True, val=True)
            image_count += 1
    print(f"Inference complete. Processed {image_count} images.")
